# ParkIQ – Exploratory Data Analysis
Bengaluru police violation dataset · Nov 2023 → Apr 2024

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'src'))

import json, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
print('imports ok')

## 1. Load raw data

In [ ]:
from parkiq.io_load import load_raw
df = load_raw()
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
print('Dtypes:')
print(df.dtypes)
print('\nNull counts:')
print(df.isnull().sum().sort_values(ascending=False))

## 2. Violation type distribution

In [ ]:
# Explode JSON-list violations into individual rows
all_violations = [v for lst in df['violation_type'] for v in lst]
vt_counts = pd.Series(all_violations).value_counts().reset_index()
vt_counts.columns = ['violation', 'count']

fig = px.bar(vt_counts, x='violation', y='count', color='count',
             color_continuous_scale='Reds', title='Violation type counts')
fig.update_xaxes(tickangle=45)
fig.show()

## 3. Parking vs non-parking split

In [ ]:
from parkiq.config import PARKING_KEYWORDS
df['is_parking'] = df['violation_type'].apply(
    lambda lst: any(str(v).upper() in PARKING_KEYWORDS for v in lst)
)
print(df['is_parking'].value_counts())
print(f'Parking share: {df["is_parking"].mean()*100:.1f}%')

## 4. Temporal analysis

In [ ]:
from parkiq.clean import clean
from parkiq.features import build_features

df_c = clean(df, parking_only=True)
df_c = build_features(df_c)
print(f'After clean: {df_c.shape}')

In [ ]:
hour_counts = df_c.groupby('hour').size().reset_index(name='count')
fig = px.bar(hour_counts, x='hour', y='count', title='Violations by hour of day',
             color='count', color_continuous_scale='OrRd')
fig.show()

In [ ]:
day_counts = df_c.groupby('day_of_week').size().reset_index(name='count')
day_counts['day'] = day_counts['day_of_week'].map(
    {0:'Mon',1:'Tue',2:'Wed',3:'Thu',4:'Fri',5:'Sat',6:'Sun'})
fig = px.bar(day_counts, x='day', y='count', title='Violations by weekday')
fig.show()

## 5. Spatial scatter (Bengaluru)

In [ ]:
sample = df_c.sample(min(5000, len(df_c)), random_state=42)
fig = px.scatter_mapbox(
    sample, lat='latitude', lon='longitude',
    color='primary_violation', hover_data=['police_station','junction_name'],
    zoom=11, mapbox_style='carto-darkmatter',
    title='Parking violations – spatial scatter (5k sample)',
    height=600,
)
fig.show()

## 6. Top police stations

In [ ]:
ps = df_c['police_station'].value_counts().head(20).reset_index()
ps.columns = ['station','count']
fig = px.bar(ps, x='station', y='count', title='Top 20 police stations by volume',
             color='count', color_continuous_scale='Blues')
fig.update_xaxes(tickangle=45)
fig.show()

## 7. Repeat offenders

In [ ]:
veh_counts = df_c['vehicle_number'].value_counts()
repeat = veh_counts[veh_counts > 1]
print(f'Unique vehicles: {len(veh_counts):,}')
print(f'Repeat offenders (seen >1×): {len(repeat):,} ({len(repeat)/len(veh_counts)*100:.1f}%)')
print('\nTop 10 repeat vehicles:')
print(repeat.head(10))

## 8. H3 hex density preview

In [ ]:
try:
    import h3
    from parkiq.geo import add_h3
    sample_h3 = add_h3(sample)
    h3_counts = sample_h3['h3_r9'].value_counts().head(20)
    print('Top H3 cells (res 9):')
    print(h3_counts)
except ImportError:
    print('h3 not installed – skipping H3 preview')

## 9. Severity weight distribution

In [ ]:
sw = df_c.groupby('primary_violation')['severity_weight'].first().sort_values(ascending=False)
fig = px.bar(sw.reset_index(), x='primary_violation', y='severity_weight',
             title='Severity weights by violation type',
             color='severity_weight', color_continuous_scale='YlOrRd')
fig.update_xaxes(tickangle=45)
fig.show()

## Summary stats

In [ ]:
print('=== ParkIQ Dataset Summary ===')
print(f'  Total records (raw):          {len(df):,}')
print(f'  Parking violations (cleaned): {len(df_c):,}')
print(f'  Date range:                   {df_c["created_datetime"].min().date()} → {df_c["created_datetime"].max().date()}')
print(f'  Police stations:              {df_c["police_station"].nunique()}')
print(f'  Unique vehicles:              {df_c["vehicle_number"].nunique():,}')
print(f'  Junction-linked violations:   {df_c["near_junction"].sum():,} ({df_c["near_junction"].mean()*100:.1f}%)')
print(f'  Repeat offenders:             {df_c["is_repeat_offender"].sum():,} ({df_c["is_repeat_offender"].mean()*100:.1f}%)')